# ANALIZA CEN NAFTNIH DERIVATOV

V tem zvezku je prestavljena analiza tega, kako se gibljejo regularne maloprodajne cene goriva v Sloveniji (bencin-95, dizel, kurilno olje) v primerjavi z gibanjem cene surove nafte na svetovnem trgu (Brent, WTI) ter gibanjem maloprodajnih cen naših sosednjih držav(Italija, Avstrija).

## 1. Uvoz knjižnic in modulov

In [ ]:
import sys
sys.path.append("src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scraper import zajemi_cene, shrani as shrani_si
from fetch_oil import zajemi_cene_nafte, shrani as shrani_nafta
from proces import preberi_podatke, zdruzi_podatke, shrani as shrani_zdruzeno
from analiza1 import drsece_povprecje, korelacija, volatilnost
from fetch_eu import prenesi_datoteko, izlusci_cene, shrani as shrani_eu
from primerjava import povprecne_cene, razlika_do_referencne, korelacijska_matrika, najvecja_razlika

sns.set_theme(style="whitegrid")


## 2. Zajem podatkov

V tem koraku zajamemo podatke iz dveh (kasneje treh) virov: regulirane
cene goriva v Sloveniji s Portala Energetika, svetovne cene surove nafte
(Brent, WTI) prek Yahoo Finance, ter kasneje še cene goriva v Avstriji in
Italiji prek uradne EU tabele. Podatke shranimo lokalno v obliki CSV
datotek, da jih ni treba vsakič znova prenašati s spleta.

In [ ]:
si_cene = zajemi_cene()
si_cene.head(10)
shrani_si(si_cene)
si_cene.head()

## 3. Obdelava in spajanje podatkov

In [ ]:
si_cene, svetovne_cene = preberi_podatke()
cene = zdruzi_podatke(si_cene, svetovne_cene)
shrani_zdruzeno(cene)
cene.head()

## 4. Gibanje cen skozi čas

Primerjava slovenske cene bencina in cene Brent nafte na istem grafu.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(cene["datum"], cene["bencin_95"], color="tab:blue", label="Bencin NMB-95 (EUR/l)")
ax1.set_ylabel("Cena bencina (EUR/l)", color="tab:blue")
ax1.set_xlabel("Datum")

ax2 = ax1.twinx()
ax2.plot(cene["datum"], cene["brent"], color="tab:orange", label="Brent (USD/sod)")
ax2.set_ylabel("Cena Brent nafte (USD/sod)", color="tab:orange")

fig.suptitle("Gibanje cene bencina v Sloveniji vs. svetovna cena nafte (Brent)")
fig.tight_layout()
plt.show()

## 5. Korelacija med cenami

Korelacija med ceno bencina in Brent nafto znaša 0.53, kar kaže na močno pozitivno povezavo. To je smiselno, saj je surova
nafta osnovna surovina za proizvodnjo bencina, cena na črpalki pa poleg
nabavne cene vključuje tudi trošarine, DDV in maržo distributerja, ki se
ne spreminjajo tako pogosto ali neposredno kot cena nafte na svetovnem trgu.
Zato povezava ni popolna (korelacija 1,0), ampak je jasno vidna.

In [ ]:
for derivat in ["bencin_95", "dizel", "kurilno_olje"]:
    korelacija1 = korelacija(cene, derivat, "brent")
    print(derivat, "- korelacija z Brent nafto:", round(korelacija1, 3))

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(data=cene, x="brent", y="bencin_95", alpha=0.5)
plt.xlabel("Cena Brent nafte (USD/sod)")
plt.ylabel("Cena bencina v Sloveniji (EUR/l)")
plt.title("Razmerje med svetovno ceno nafte in ceno bencina")
plt.show()

## 6. Drseča povprečja in volatilnost

Drseče povprečje (30 dni) lepo zgladi kratkoročna nihanja cene in pokaže
dolgoročni trend. Iz grafa je razvidno, da je bila cena bencina najbolj
volatilna v letih 2022 in 2026,kar sovpada z začetkom vojne v Ukrajini v letu 2022 in vojne v Iranu in zaprtjem Hormuške ožine, v letu 2026, kar pa je trenutno še vedno aktualno. V mirnejših obdobjih se volatilnost giblje med 1.4 in 1.5.

In [ ]:
cene = drsece_povprecje(cene, "bencin_95", okno=30)
cene["volatilnost_brent"] = volatilnost(cene, "brent", okno=30)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cene["datum"], cene["bencin_95"], label="Cena bencina", alpha=0.4)
ax.plot(cene["datum"], cene["bencin_95_drsece_30d"], label="30-dnevno drseče povprečje", linewidth=2)
ax.set_xlabel("Datum")
ax.set_ylabel("EUR/liter")
ax.legend()
ax.set_title("Cena bencina in drseče povprečje")
plt.show()

## 7. Primerjava s sosednjimi državami (Slovenija, Avstrija, Italija)

Iz primerjave s sosednjima državama je razvidno, da je Slovenija skozi
celotno obravnavano obdobje v povprečju najcenejša izmed treh držav, čeprav
se z Avstrijo večkrat "zamenjata" - v posameznih obdobjih je bila cena v
Avstriji nižja od slovenske. Italija je ves čas jasno najdražja. Razlike
med državami so verjetno posledica različnih stopenj trošarin in DDV, ki
jih vsaka država določa samostojno, medtem ko je osnovna nabavna cena
(surova nafta) za vse enaka.

In [ ]:

prenesi_datoteko()
eu_cene = izlusci_cene()
shrani_eu(eu_cene)
eu_cene.tail()

### Povprečne cene po državah

In [ ]:
stolpci_bencin = ["Slovenija_bencin", "Avstrija_bencin", "Italija_bencin"]

povprecja = povprecne_cene(eu_cene, stolpci_bencin)

plt.figure(figsize=(7, 5))
povprecja.plot(kind="bar", color=["tab:blue", "tab:orange", "tab:green"])
plt.ylabel("Povprečna cena bencina (EUR/1000 l)")
plt.title("Povprečna cena bencina po državah")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Gibanje cen skozi čas - primerjava treh držav

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(eu_cene["datum"], eu_cene["Slovenija_bencin"], label="Slovenija")
plt.plot(eu_cene["datum"], eu_cene["Avstrija_bencin"], label="Avstrija")
plt.plot(eu_cene["datum"], eu_cene["Italija_bencin"], label="Italija")
plt.xlabel("Datum")
plt.ylabel("Cena bencina (EUR/1000 l)")
plt.title("Gibanje cene bencina - Slovenija, Avstrija, Italija")
plt.legend()
plt.tight_layout()
plt.show()

### Razlika v ceni glede na Slovenijo

In [ ]:
razlike = razlika_do_referencne(eu_cene, stolpci_bencin, "Slovenija_bencin")

plt.figure(figsize=(12, 5))
plt.plot(razlike["datum"], razlike["Avstrija_bencin_razlika"], label="Avstrija - Slovenija")
plt.plot(razlike["datum"], razlike["Italija_bencin_razlika"], label="Italija - Slovenija")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Datum")
plt.ylabel("Razlika v ceni (EUR/1000 l)")
plt.title("Razlika v ceni bencina glede na Slovenijo")
plt.legend()
plt.tight_layout()
plt.show()

### Korelacija med državami

In [ ]:
korelacije = korelacijska_matrika(eu_cene, stolpci_bencin)
print(korelacije)

datum_najvecje, velikost_razlike = najvecja_razlika(eu_cene, "Avstrija_bencin", "Slovenija_bencin")
print("Največja razlika Avstrija-Slovenija je bila", datum_najvecje, "in je znašala", round(velikost_razlike, 2), "EUR/1000l")

## 8. Zaključek





Analiza je pokazala, da se cena bencina v Sloveniji precej giblje skupaj
s svetovno ceno nafte (Brent), povezava pa ni popolna, ker končna cena
poleg nabavne cene vključuje tudi davke in maržo distributerja, ki se ne
spreminjajo tako hitro kot cena surove nafte. Drseča povprečja so pokazala
obdobja, ko je cena bolj nihala, zanimivo pa je bilo tudi odkritje, da
kurilno olje leta 2020 sploh še ni bilo regulirano, cene bencina in dizla
pa so bile takrat fiksne.

Pri primerjavi s sosednjima državama se je Slovenija izkazala za eno
cenejših, z Avstrijo se v posameznih obdobjih izmenjujeta za najnižjo
ceno, Italija pa je ves čas dražja - kar je najverjetneje posledica
različnih davkov v posameznih državah, saj je cena surove nafte za vse
enaka.